# Module 6 Assignment: Give Your Persona Agent More Tools

In Module 5 you built a Persona Agent with one tool. This assignment makes the same jump Lab 5
made on top of Lab 4: your agent goes from one tool to three, and `chat()` has to learn to route
between them instead of assuming.

You'll reuse your own Module 5 code at almost every step, plus the routing pattern from Lab 5.
This assignment is about **extending** what you already built, not writing it from scratch.

**What you'll build:**
1. Your persona, notes, and first tool, carried over from Module 5
2. A second tool, backed by a free, no-key external API
3. A third tool of your own design
4. JSON descriptions for all three tools
5. An updated system prompt that tells the model when to use each tool
6. A `call_tool(name, arguments)` dispatcher, exactly like Lab 5's
7. A full agent loop that routes across all three tools

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you can find where Lab 5 made this same change to Lab 4, you can make it here too.

## 0. Install the packages (run once)

Same as Module 5, plus one new package: `requests`, for calling your new tool's API.

In [ ]:
!uv pip install --upgrade openai pypdf gradio python-dotenv requests

## 1. Imports

**TODO:** Copy the imports from your Module 5 assignment, plus `requests` for calling an external
API. Add any other standard library module your second or third tool needs (for example `html`,
if the API you pick returns HTML-escaped text the way Open Trivia Database does).

In [ ]:
from dotenv import load_dotenv
import os
import sys
import json

import requests
from openai import AzureOpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr


## 2. Connect to Azure OpenAI via APIM

Identical to every previous lab and assignment. Copy this cell directly from Module 5.

In [ ]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")

if not all([azure_endpoint, azure_api_key]):
    sys.exit("Missing settings. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file.")

print(f"Azure OpenAI key exists and begins {azure_api_key[:8]}")

# The deployment we're using for this whole assignment - fixed, do not change it
DEPLOYMENT = "gpt-5.4-ptu"

azure_client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version="2024-10-21",
)

## 3. Recreate your persona, notes, and first tool from Module 5

Bring forward everything you already built: your `persona`, your `notes/notes.pdf` loading code,
your one tool function from Module 5, and that tool's JSON description. Nothing here needs to
change, this is your starting point.

**TODO:** Paste in, from your Module 5 assignment:
- Your `persona` variable
- Your `PdfReader("notes/notes.pdf")` loading code that builds `notes`
- Your Module 5 tool function (rename it if you like, but keep track of its name)
- That tool's JSON schema dictionary

**Hint:** copy your `notes/notes.pdf` file into this assignment's folder too, in a `notes`
subfolder, exactly like Lab 5 did with Lab 4's notes file.

In [ ]:
persona = (
    "Root, a graybeard Linux systems administrator with twenty years of uptime behind him. "
    "Root talks in short, clipped lines, never uses two words where one will do, and calls the "
    "student \"operator\". He has no patience for guessing and says so plainly, but he is never "
    "cruel - when you miss something, he tells you exactly which command or man page to go read."
)

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

# Tool 1, carried forward from Module 5: append a missed command or concept to a review file.
WEAK_COMMANDS_FILE = "weak_commands.txt"


def log_weak_command_tool(command):
    print(f"Tool called to log a weak spot: {command}")
    with open(WEAK_COMMANDS_FILE, "a", encoding="utf-8") as f:
        f.write(command + "\n")
    return "Weak spot logged"


print(persona)
print(notes[:300], "...")


In [ ]:
log_weak_command_tool_json = {
    "name": "log_weak_command_tool",
    "description": (
        "Use this tool to record a Linux command or command-line concept the operator got wrong "
        "or said they did not know, so it can be reviewed later. Call it once per missed item."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "command": {
                "type": "string",
                "description": "The command or concept the operator missed, e.g. 'chmod octal notation' or 'kill -9 vs SIGTERM'",
            }
        },
        "required": ["command"],
        "additionalProperties": False,
    },
}


## 4. New tool: something backed by a free API

Design a **second** tool for your Persona Agent, this one calling a free, no-key external API,
the same way Lab 5's `fetch_bonus_question_tool` calls the Open Trivia Database.

**Ideas, pick whichever fits your topic, or find your own:**
- A trivia-friendly agent could call the Open Trivia Database for a bonus question
- A weather-aware topic could call [Open-Meteo](https://open-meteo.com), a free weather API that
  needs no key
- A language-learning agent could call the [Free Dictionary API](https://dictionaryapi.dev) to
  define a word, though it is sometimes slow or briefly unavailable, so handle that the same way
  the next tip describes
- Any other free, no-key API relevant to your topic works too, just check it does not require
  signup or a paid key

**TODO:** Write your tool function below. It should make one `requests.get(...)` call inside a
`try`/`except` that catches `requests.exceptions.RequestException`, so a slow or unavailable API
returns a short, friendly string instead of crashing your notebook, exactly like Lab 5's tools
do.

In [ ]:
# Tool 2: a free, no-key external API. Wikipedia's REST summary endpoint needs no signup and no
# key, and it fits Root's habit of pointing the operator at something to go read.
WIKIPEDIA_SUMMARY_URL = "https://en.wikipedia.org/api/rest_v1/page/summary/"


def lookup_command_reference_tool(topic):
    print(f"Tool called to look up a reference for: {topic}")
    title = requests.utils.quote(topic.strip().replace(" ", "_"))

    try:
        response = requests.get(
            WIKIPEDIA_SUMMARY_URL + title,
            headers={"User-Agent": "RootStudyAgent/1.0 (coursework)"},
            timeout=20,
        )
        data = response.json()
    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"lookup_command_reference_tool network error: {e}")
        return f"Reference lookup is down. Tell the operator to run 'man {topic}' instead."

    if response.status_code == 404:
        return f"No reference article found for '{topic}'. Tell the operator to check 'man {topic}'."
    if "extract" not in data or not data["extract"].strip():
        return f"Reference lookup returned nothing useful for '{topic}'. Fall back to 'man {topic}'."

    extract = " ".join(data["extract"].split())
    if len(extract) > 400:
        extract = extract[:400].rsplit(" ", 1)[0] + "..."
    return f"Reference for {data.get('title', topic)}: {extract}"


In [ ]:
lookup_command_reference_tool("grep")  # quick manual test of the API-backed tool


## 5. New tool: one more tool of your own design

Design a **third** tool. It can be another local tool in the spirit of Module 5 (a running score
tracker, a bookmarking tool for topics that need review, or an idea of your own), or a second
external API call if you would rather. Either is fine, as long as it does something the first two
tools do not already cover.

**TODO:** Write your third tool function below.

In [ ]:
# Tool 3: the counterpart to tool 1. Tool 1 only ever writes to the review file, which makes the
# agent's memory write-only. This reads it back so Root can open with "last time you missed these".


def review_weak_commands_tool(count=None):
    print(f"Tool called to review logged weak spots (count={count})")
    try:
        with open(WEAK_COMMANDS_FILE, "r", encoding="utf-8") as f:
            weak = [line.strip() for line in f if line.strip()]
    except FileNotFoundError:
        weak = []

    if not weak:
        return "Nothing logged yet. Operator has a clean sheet."

    n = count if isinstance(count, int) and count > 0 else 5
    recent = weak[-n:]
    return f"{len(weak)} weak spot(s) logged. Most recent {len(recent)}: " + "; ".join(recent)


In [ ]:
review_weak_commands_tool(3)  # quick manual test of the third tool


## 6. Describe all three tools to the model with JSON

Same idea as Module 5 and Lab 5, just three schemas now. Remember an argument can be optional (an
empty `"required"` list), exactly like Lab 5's `fetch_bonus_question_tool`, if that fits your
tool.

In [ ]:
lookup_command_reference_tool_json = {
    "name": "lookup_command_reference_tool",
    "description": (
        "Use this tool to fetch a short external reference summary for a Linux command or "
        "command-line concept, when the operator asks what something is, asks for more detail "
        "than the notes give, or when you want to back up an explanation with a source."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "description": "The command or concept to look up, e.g. 'grep', 'sudo' or 'symbolic link'",
            }
        },
        "required": ["topic"],
        "additionalProperties": False,
    },
}

review_weak_commands_tool_json = {
    "name": "review_weak_commands_tool",
    "description": (
        "Use this tool when the operator asks how they are doing, what they have missed so far, "
        "or asks to review their weak spots. Returns the most recently logged weak spots."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "count": {
                "type": "integer",
                "description": "How many of the most recent weak spots to return. Defaults to 5 if left out.",
            }
        },
        # count is optional - leaving it out of "required" lets the model call this with no arguments
        "required": [],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": log_weak_command_tool_json},
    {"type": "function", "function": lookup_command_reference_tool_json},
    {"type": "function", "function": review_weak_commands_tool_json},
]


## 7. Update your system prompt for three tools

**TODO:** Build your `system_prompt` f-string the same way you did in Module 5, still using
`persona` and `notes`, but now with three rules telling the model when to reach for each tool,
the same way Lab 5's system prompt added two new sentences to Lab 4's.

In [ ]:
system_prompt = f"""
# Your role

You are {persona}

You quiz the operator on the Linux and command-line study notes below.

# Context

Here are the study notes you are quizzing the operator on:

{notes}

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the operator's answer
before asking the next question.

After the operator answers, say clearly whether they were right or wrong, and give a one-sentence
explanation either way. Stay in character the whole time: short, clipped lines, no filler.

If the operator's answer was wrong, or they say "I don't know", you must record the command or
concept they missed by calling the log_weak_command_tool tool before you move on to the next
question. If they admit to missing several things at once, call the tool once for each of them.

If the operator asks what a command or concept actually is, wants more detail than the notes give,
or you want to back an explanation with a source, call the lookup_command_reference_tool tool with
that command or concept and work the summary it returns into your answer in your own clipped voice.

If the operator asks how they are doing, what they have missed so far, or asks to review their weak
spots, call the review_weak_commands_tool tool and report back what it returns.

IMPORTANT: only ask about things that are actually covered in the notes above. If the operator
asks about something outside the notes, say plainly that it is not in this study set, and point
them at `man` for it instead of answering from memory.
"""

display(Markdown(system_prompt))


## 8. The routing problem: which tool did the model actually call?

With three tools, `chat()` can no longer assume. It has to check `tool_call.function.name` and
call the matching Python function.

**TODO:** Copy the `call_tool(name, arguments)` dispatcher pattern from Lab 5, Step 9, but route
across your own three tool names instead.

In [ ]:
def call_tool(name, arguments):
    if name == "log_weak_command_tool":
        return log_weak_command_tool(arguments.get("command"))
    elif name == "lookup_command_reference_tool":
        return lookup_command_reference_tool(arguments.get("topic"))
    elif name == "review_weak_commands_tool":
        return review_weak_commands_tool(arguments.get("count"))
    else:
        return f"Unknown tool: {name}"


## 9. Update `chat()` to route across all three tools

**TODO:** Copy the finished agent loop from your Module 5 assignment. The `while` loop and the
`for tool_call in message_obj.tool_calls` loop do not change at all, the only new line is a call
to `call_tool(...)` in place of your old single hardcoded tool call.

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        messages.append(message_obj)
        for tool_call in message_obj.tool_calls:
            name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            result = call_tool(name, arguments)
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    return response.choices[0].message.content


## 10. Try your multi-tool Persona Agent

Launch the chat interface and try to trigger all three tools in one session: something that
should hit your original Module 5 tool, something that should hit your new API-backed tool, and
something that should hit your third tool. Watch the terminal or output to see which tool fires
each time. Interrupt the cell when you're done.

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

## Reflection questions

Answer these in a new Markdown cell below.

1. What are your three tools, and what does each one do? Why did you choose the second and third
   tools you designed?
2. Compare this assignment to Module 5. What was exactly the same as going from one tool to more
   than one? What, if anything, surprised you?
3. Walk through what `call_tool` actually does the first time your agent calls your new
   API-backed tool. What would break if you deleted the `elif` branch for that tool?
4. Open Lab 5's system prompt and compare it to yours. Both add rules for two new tools, using
   only plain English sentences and no code changes to the agent loop. Why does that work, in
   terms of what a system prompt actually is?

## Reflection answers

**1. What are your three tools, and what does each one do? Why did you choose the second and third tools you designed?**

The agent is still Root, the terse sysadmin from Module 5, quizzing me on my Linux and
command-line notes. The three tools are:

- `log_weak_command_tool(command)` - carried straight over from Module 5. Appends a command or
  concept I missed to `weak_commands.txt` and prints that it did.
- `lookup_command_reference_tool(topic)` - the new API-backed tool. It hits Wikipedia's REST
  summary endpoint (`/api/rest_v1/page/summary/<title>`), which needs no key and no signup, and
  returns a trimmed one-paragraph summary of a command or concept.
- `review_weak_commands_tool(count)` - reads `weak_commands.txt` back and returns the most recent
  N entries, defaulting to 5 if the model doesn't pass a count.

I picked the second one because of what Root is supposed to do. His whole character is "go read
the man page" - he's a pointer to documentation, not an encyclopedia. A tool that fetches a real
external description of a command is the most in-character thing I could give him, and it does
something the notes genuinely can't: the notes are a fixed 6KB of text I wrote, so anything I
didn't cover is simply unavailable to the agent otherwise. I considered Open Trivia DB like Lab 5
and `cheat.sh`, which would have been the most thematically perfect of all, but `cheat.sh` returned
HTTP 429 when I tested it, so it would have spent most of its life in the except branch. Wikipedia
answered in under half a second every time I hit it.

The third one is the tool I said I'd build in Module 5's reflection, so this was mostly keeping my
word. The point I made then still stands: with only `log_weak_command_tool`, the agent's memory is
write-only. It can record that I blanked on `chmod` octal notation and then never mention it again.
Pairing a reader with the writer is what turns the file from a dead log into something the agent
can actually open a session with. It's also distinct from the other two in the way the rubric asks
for - tool 1 writes locally, tool 2 reads remotely, tool 3 reads locally.

**2. Compare this assignment to Module 5. What was exactly the same as going from one tool to more than one? What, if anything, surprised you?**

The agent loop was exactly the same, and that's the headline. I did not touch the `while` or the
`for` in `chat()`. The only line that changed is the one that used to read
`result = log_weak_command_tool(command)` and now reads `result = call_tool(name, arguments)`. The
client setup, the PDF loading, the `role: "tool"` message shape, the Gradio line, and the schema
dict shape are all identical too. Going from one tool to three was almost entirely additive.

Two things surprised me. First, how much of the work was writing English rather than Python. The
dispatcher is nine lines; the schema descriptions and the three new system prompt sentences took
longer to get right, because those are what actually decide whether the correct tool fires. Second,
and more concretely: once a tool reaches out to the network, "returns a short string" starts doing
real work. My API tool has four separate return paths - network failure, 404, an empty `extract`,
and success - and three of them are failures. The success path also needed trimming, because a full
Wikipedia summary can be several hundred words and all of that would get pasted into the message
history on every call, in an agent whose system prompt already carries the entire notes file. A
local tool like `log_weak_command_tool` never made me think about any of that.

**3. Walk through what `call_tool` actually does the first time your agent calls your new API-backed tool. What would break if you deleted the `elif` branch for that tool?**

Say I ask "what actually is grep?". The model returns with `finish_reason == "tool_calls"`, so the
`while` loop opens. `message_obj.tool_calls[0].function.name` is the string
`"lookup_command_reference_tool"` and `.function.arguments` is the JSON *string*
`'{"topic": "grep"}'`, which `json.loads` turns into the dict `{"topic": "grep"}`. Both go into
`call_tool`. The first `if` compares against `"log_weak_command_tool"` and fails, the first `elif`
matches, and it calls `lookup_command_reference_tool(arguments.get("topic"))` - so the actual
`requests.get` to Wikipedia happens inside the dispatcher branch, on the string the model chose.
The returned string goes back into `messages` as a `"tool"` message tagged with that call's
`tool_call.id`, the model is called again with the full history, and this time it answers in Root's
voice using the summary.

Deleting that `elif` wouldn't raise anything, which is the part worth noticing. `name` would fall
past `log_weak_command_tool`, past `review_weak_commands_tool`, and land in the `else`, which
returns the string `"Unknown tool: lookup_command_reference_tool"`. That gets appended as a
perfectly well-formed tool result, the loop exits normally, and the model writes an answer on top
of it - probably apologising that it can't look that up, or quietly answering from its own training
data instead. No traceback, no error in the UI, just a silently degraded agent. The schema would
still advertise the tool to the model, so it would keep calling it and keep getting nothing. That
`else` branch is doing more than tidiness: without it the mismatch would be a `TypeError` on
`None`, and honestly that might be easier to spot.

**4. Open Lab 5's system prompt and compare it to yours. Both add rules for two new tools, using only plain English sentences and no code changes to the agent loop. Why does that work, in terms of what a system prompt actually is?**

Lab 5 added two sentences to Lab 4's prompt - one for `fetch_bonus_question_tool`, one for
`check_progress_tool` - and I added two to mine, one for the reference lookup and one for the
review tool. Neither of us changed the loop, because the system prompt is not configuration that
the loop reads. It's just the first message in the `messages` list, ordinary text the model
conditions on. My code never parses it. `chat()` would run identically if the whole Rules section
said nothing about tools at all.

What's actually happening is that two separate channels both describe the same three functions, and
they do different jobs. The `tools` list is the machine-readable half: it tells the model the names,
the argument names, and the types, and it's what constrains the JSON the model emits. The system
prompt is the policy half: it tells the model *when* a situation calls for each one. The schema can
say `topic` is a string; only the prompt can say "when the operator asks what something is, reach
for this." So adding a tool is three independent edits - a Python function, a schema in `tools`, a
sentence of policy - and only the function and the schema are code. The dispatcher is the one place
the two channels have to agree, since it's matching on the same name string the schema declares.

The honest limit is the same one I noted in Module 5. These rules are instructions, not guarantees.
Nothing in `chat()` checks that the model called the right tool, or any tool. If it decides to
answer "what is grep" from memory instead of calling `lookup_command_reference_tool`, my code has
no idea and no objection. Plain English is enough to *steer* tool choice precisely because the model
is the thing doing the choosing - and that's also exactly why it can't enforce it.


## Wrap-up

Before submitting:

- Make sure your `.env` file is **not** included in what you submit, it contains your API key.
- Run the notebook top to bottom one more time to confirm it works cleanly.
- Answer all four reflection questions.
- Rename this file to include your name, for example `Module6Assignment_JaneDoe.ipynb`, and
  submit it.

You've now made the same jump Lab 5 made: one tool to several, with a real dispatcher and a loop
that already scaled to handle it.